# Orphadata Science — Rare Disease Data Ingestion

**Orphadata Science** (formerly Orphadata) is the open, structured dataset arm of **Orphanet**, the reference portal for rare diseases and orphan drugs. It is one of the ELIXIR Core Data Resources and is maintained at INSERM US14 (Paris). Orphadata publishes expertly curated nomenclatures, classifications, and annotations for roughly **6,000 rare diseases**, and is the single most comprehensive, cross-referenced source of rare-disease knowledge available for research.

Every rare disorder carries a stable **ORPHAcode** that is cross-referenced to ICD-10/ICD-11, OMIM, MeSH, UMLS, MedDRA, GARD, and SNOMED-CT — making it an essential bridge between clinical vocabularies and molecular-biology databases.

Key data products (XML, downloadable from https://www.orphadata.com/):

| Product | File | Content |
|---|---|---|
| 1 | `en_product1.xml` | Rare disease cross-referencing (ORPHAcode ↔ ICD-10, OMIM, MeSH, UMLS, MedDRA, GARD) |
| 3 | `en_product3_<N>.xml` | Disorder classifications by medical speciality |
| 4 | `en_product4.xml` | Phenotypes associated with rare diseases (HPO terms + frequency) |
| 6 | `en_product6.xml` | Genes associated with rare diseases (HGNC, Ensembl, OMIM, SwissProt) |
| 7 | `en_product7.xml` | Linearisation of rare diseases (ICD-11-style) |
| 9 | `en_product9_ages.xml` | Age of onset and age of death |
| 9 | `en_product9_prev.xml` | Epidemiological data — prevalence, incidence, number born |

**Bulk downloads:** `https://www.orphadata.com/data/xml/<file>`

**License:** Creative Commons Attribution 4.0 (CC-BY 4.0).

**Reference:** Weinreich et al. (2008), *Orphanet: a European database for rare diseases*. Ned Tijdschr Geneeskd. See also https://www.orphadata.com/ for the full documentation.

In [ ]:
import requests
import time
from pathlib import Path
import xml.etree.ElementTree as ET

import polars as pl

# TODO

* [x] **Ingest data**
    * [x] Download Orphadata XML bulk files with caching (`en_product1.xml`, `en_product6.xml`)
    * [x] Parse the cross-reference product into a Polars DataFrame
    * [x] Parse the disorder–gene product into a Polars DataFrame
    * [x] Save data to `data/` with caching
* [ ] **Explore and clean**
    * [ ] Summarise entry counts by disorder type and disorder group
    * [ ] Inspect completeness of cross-reference sources (ICD-10, OMIM, MeSH, UMLS, MedDRA, GARD)
    * [ ] Handle deprecated / obsolete ORPHAcodes and duplicates
    * [ ] Join disorder-level and gene-level tables on ORPHAcode
* [ ] **Analysis**
    * [ ] Distribution of number of genes per disorder — long-tail or power-law?
    * [ ] Most "cross-referenced" disorders (mapping-completeness ranking)
    * [ ] Cross-reference overlap matrix (e.g. what fraction of OMIM-mapped disorders also map to UMLS?)
    * [ ] Pleiotropy: genes implicated in many rare diseases
* [ ] **Visualization**
    * [ ] Bar chart of disorders per disorder group
    * [ ] Histogram of gene-counts per disorder (log-scale)
    * [ ] Upset plot of cross-reference source overlap
    * [ ] Network graph of highly pleiotropic genes ↔ disorders
* [ ] **Statistical analysis**
    * [ ] Power-law fit (maximum-likelihood, Clauset-Shalizi-Newman) for gene-per-disorder distribution
    * [ ] Discuss sampling bias — curation intensity varies by disease area
    * [ ] Bootstrap confidence intervals for prevalence-weighted statistics
    * [ ] Multiple-testing correction when screening thousands of gene–disease associations

## 1. Ingest Data

### 1.1 Download Orphadata XML Bulk Files

Orphadata publishes a stable set of XML products at `https://www.orphadata.com/data/xml/`. Each file is small enough (a few MB) to download in one request, but we still stream to disk and cache locally so re-runs are instant and network-free.

In [ ]:
ORPHADATA_BASE = "https://www.orphadata.com/data/xml"
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)  # create data/ directory if it does not exist

# Products of interest for this notebook.
# en_product1.xml - disorder cross-referencing (ICD-10, OMIM, MeSH, UMLS, ...)
# en_product6.xml - disorder <-> gene associations (HGNC, Ensembl, SwissProt)
PRODUCTS = {
    "cross_refs": "en_product1.xml",
    "disorder_gene": "en_product6.xml",
}


def download_orphadata(filename: str, cache_dir: Path = DATA_DIR) -> Path:
    """
    Download an Orphadata XML product, caching it on disk.

    Parameters
    ----------
    filename : str
        Basename of the product file, e.g. ``"en_product1.xml"``.
    cache_dir : Path, default ``DATA_DIR``
        Directory in which the file is cached.

    Returns
    -------
    Path
        Local path to the downloaded (or cached) XML file.

    Notes
    -----
    Orphadata XMLs are typically 3-15 MB uncompressed. We stream in 64 KB
    chunks to keep peak memory flat and sleep briefly after a real download
    to be a polite client. A cache hit is silent and instantaneous.
    """
    url = f"{ORPHADATA_BASE}/{filename}"
    out_path = cache_dir / filename

    if out_path.exists():
        # Already cached locally - no network call required.
        size_mb = out_path.stat().st_size / 1_048_576
        print(f"Cache hit: {out_path}  ({size_mb:.1f} MB)")
        return out_path

    print(f"Downloading: {url}")
    with requests.get(url, stream=True, timeout=180) as resp:
        resp.raise_for_status()
        # Stream to disk in 64 KB chunks so memory stays flat for large files.
        with out_path.open("wb") as fh:
            for chunk in resp.iter_content(chunk_size=65_536):
                fh.write(chunk)

    size_mb = out_path.stat().st_size / 1_048_576
    print(f"Saved to  : {out_path}  ({size_mb:.1f} MB)")
    time.sleep(0.5)  # polite pause between consecutive downloads
    return out_path


# Download both products (cached on re-runs).
xref_path = download_orphadata(PRODUCTS["cross_refs"])
gene_path = download_orphadata(PRODUCTS["disorder_gene"])

### 1.2 Parse `en_product1.xml` — Disorder Cross-References

The `en_product1.xml` file has the following nested structure (simplified):

```
<JDBOR date="...">
  <DisorderList count="...">
    <Disorder id="...">
      <OrphaCode>166024</OrphaCode>
      <Name lang="en">Multiple epiphyseal dysplasia, Al-Gazali type</Name>
      <DisorderType><Name lang="en">Disease</Name></DisorderType>
      <DisorderGroup><Name lang="en">Disorder</Name></DisorderGroup>
      <ExpertLink lang="en">http://www.orpha.net/...</ExpertLink>
      <ExternalReferenceList count="N">
        <ExternalReference id="...">
          <Source>ICD-10</Source>
          <Reference>Q77.3</Reference>
          <DisorderMappingRelation><Name lang="en">NTBT ...</Name></DisorderMappingRelation>
        </ExternalReference>
        ...
      </ExternalReferenceList>
    </Disorder>
    ...
  </DisorderList>
</JDBOR>
```

Because each disorder can carry many external references, we emit **one row per (disorder, cross-reference) pair** — a classic tidy long-format that is easy to join, group, or pivot downstream.

In [ ]:
def _text(elem: ET.Element | None, path: str | None = None) -> str | None:
    """
    Safely extract stripped text from an XML element (or a child by path).

    Parameters
    ----------
    elem : xml.etree.ElementTree.Element or None
        Parent element; returns ``None`` if this is already ``None``.
    path : str, optional
        Relative XPath to descend into before reading text. If ``None`` the
        text of ``elem`` itself is returned.

    Returns
    -------
    str or None
        Whitespace-stripped text content, or ``None`` if the element or its
        text is missing / empty.
    """
    if elem is None:
        return None
    target = elem if path is None else elem.find(path)
    if target is None or target.text is None:
        return None
    t = target.text.strip()
    return t or None


def parse_product1(xml_path: Path) -> pl.DataFrame:
    """
    Parse Orphadata ``en_product1.xml`` into a tidy long-format DataFrame.

    Each row represents one (disorder, external-reference) pair. Disorders
    with zero external references still emit a single row with null
    cross-reference fields so they are not silently dropped.

    Parameters
    ----------
    xml_path : Path
        Path to the downloaded ``en_product1.xml`` file.

    Returns
    -------
    pl.DataFrame
        Columns:
        - ``orpha_code``    : Int64, stable disorder ID
        - ``disorder_name`` : Utf8, canonical English name
        - ``disorder_type`` : Categorical, e.g. "Disease", "Malformation syndrome"
        - ``disorder_group``: Categorical, e.g. "Disorder", "Group of disorders"
        - ``expert_link``   : Utf8, Orphanet portal URL
        - ``xref_source``   : Categorical, e.g. "ICD-10", "OMIM", "MeSH", "UMLS"
        - ``xref_id``       : Utf8, the external identifier string
        - ``xref_relation`` : Utf8, mapping relation ("E - Exact", "NTBT - ...")
    """
    # ElementTree.parse loads the whole tree; en_product1 is only a few MB
    # so this is fine. For very large files we would use iterparse + clear().
    tree = ET.parse(xml_path)
    root = tree.getroot()

    rows: list[dict] = []
    # Every disorder sits under JDBOR > DisorderList > Disorder
    for disorder in root.iterfind(".//DisorderList/Disorder"):
        orpha_code = _text(disorder, "OrphaCode")
        disorder_name = _text(disorder, "Name")
        disorder_type = _text(disorder, "DisorderType/Name")
        disorder_group = _text(disorder, "DisorderGroup/Name")
        expert_link = _text(disorder, "ExpertLink")

        xrefs = disorder.findall("ExternalReferenceList/ExternalReference")
        if not xrefs:
            # Keep disorders with no cross-references as a single null-xref row.
            rows.append({
                "orpha_code": orpha_code,
                "disorder_name": disorder_name,
                "disorder_type": disorder_type,
                "disorder_group": disorder_group,
                "expert_link": expert_link,
                "xref_source": None,
                "xref_id": None,
                "xref_relation": None,
            })
            continue

        for xref in xrefs:
            rows.append({
                "orpha_code": orpha_code,
                "disorder_name": disorder_name,
                "disorder_type": disorder_type,
                "disorder_group": disorder_group,
                "expert_link": expert_link,
                "xref_source": _text(xref, "Source"),
                "xref_id": _text(xref, "Reference"),
                "xref_relation": _text(xref, "DisorderMappingRelation/Name"),
            })

    # Construct the DataFrame with all columns as Utf8 first, then cast.
    df = pl.DataFrame(rows)

    # orpha_code is numeric and stable; cast to Int64 for efficient joining.
    # disorder_type / group / xref_source have low cardinality -> Categorical.
    df = df.with_columns(
        pl.col("orpha_code").cast(pl.Int64, strict=False),
        pl.col("disorder_type").cast(pl.Categorical),
        pl.col("disorder_group").cast(pl.Categorical),
        pl.col("xref_source").cast(pl.Categorical),
    )

    return df


xref_df = parse_product1(xref_path)

# Number of unique disorders vs. number of (disorder, xref) rows.
n_disorders = xref_df.select(pl.col("orpha_code").n_unique()).item()
print(f"Disorders parsed : {n_disorders:,}")
print(f"Total rows       : {xref_df.shape[0]:,} (one per disorder-xref pair)")
print("\nSchema:")
for col, dtype in xref_df.schema.items():
    print(f"  {col:<16} {dtype}")
xref_df.head(5)

### 1.3 Parse `en_product6.xml` — Disorder ↔ Gene Associations

`en_product6.xml` records which genes are implicated in which rare diseases, with evidence sources and cross-references to the major gene databases. Schema (simplified):

```
<JDBOR date="...">
  <DisorderList count="...">
    <Disorder id="...">
      <OrphaCode>166024</OrphaCode>
      <Name lang="en">...</Name>
      <DisorderGeneAssociationList count="N">
        <DisorderGeneAssociation>
          <SourceOfValidation>...</SourceOfValidation>
          <Gene id="...">
            <Symbol>COL11A2</Symbol>
            <Name lang="en">collagen type XI alpha 2 chain</Name>
            <ExternalReferenceList>
              <ExternalReference><Source>HGNC</Source><Reference>2187</Reference></ExternalReference>
              <ExternalReference><Source>Ensembl</Source><Reference>ENSG00000204248</Reference></ExternalReference>
              <ExternalReference><Source>OMIM</Source><Reference>120290</Reference></ExternalReference>
              <ExternalReference><Source>SwissProt</Source><Reference>P13942</Reference></ExternalReference>
              ...
            </ExternalReferenceList>
          </Gene>
          <DisorderGeneAssociationType><Name lang="en">Disease-causing germline mutation(s) in</Name></DisorderGeneAssociationType>
          <DisorderGeneAssociationStatus><Name lang="en">Assessed</Name></DisorderGeneAssociationStatus>
        </DisorderGeneAssociation>
        ...
      </DisorderGeneAssociationList>
    </Disorder>
  </DisorderList>
</JDBOR>
```

Again we flatten to long-format: **one row per (disorder, gene, association-type) triple**. Gene-level cross-references are pivoted into dedicated columns (`hgnc_id`, `ensembl_id`, `omim_id`, `swissprot_id`) because their cardinality is known in advance.

In [ ]:
# Which gene-level xref sources we want to pivot into dedicated columns.
# Anything not in this set is ignored (rare sources like Reactome appear
# occasionally and would inflate the column count for little analytic value).
GENE_XREF_SOURCES = {
    "HGNC": "hgnc_id",
    "Ensembl": "ensembl_id",
    "OMIM": "omim_id",
    "SwissProt": "swissprot_id",
    "Genatlas": "genatlas_id",
    "Reactome": "reactome_id",
    "IUPHAR": "iuphar_id",
}


def parse_product6(xml_path: Path) -> pl.DataFrame:
    """
    Parse Orphadata ``en_product6.xml`` into a long-format DataFrame of
    disorder-gene associations.

    Parameters
    ----------
    xml_path : Path
        Path to the downloaded ``en_product6.xml`` file.

    Returns
    -------
    pl.DataFrame
        One row per (disorder, gene, association-type) triple. Columns:
        - ``orpha_code``     : Int64
        - ``disorder_name``  : Utf8
        - ``gene_symbol``    : Utf8  (HGNC-approved symbol)
        - ``gene_name``      : Utf8
        - ``association_type``   : Categorical (e.g. "Disease-causing germline mutation(s) in")
        - ``association_status`` : Categorical ("Assessed" vs "Not yet assessed")
        - ``source_of_validation``: Utf8 (PubMed IDs or curator notes)
        - ``hgnc_id``, ``ensembl_id``, ``omim_id``, ``swissprot_id``,
          ``genatlas_id``, ``reactome_id``, ``iuphar_id`` : Utf8

    Notes
    -----
    A single gene can be linked to a disorder through multiple
    association types (e.g. "Disease-causing" and "Candidate gene
    tested"). Each is preserved as a separate row.
    """
    tree = ET.parse(xml_path)
    root = tree.getroot()

    rows: list[dict] = []

    for disorder in root.iterfind(".//DisorderList/Disorder"):
        orpha_code = _text(disorder, "OrphaCode")
        disorder_name = _text(disorder, "Name")

        assocs = disorder.findall("DisorderGeneAssociationList/DisorderGeneAssociation")
        if not assocs:
            # Many disorders have no known causal gene; skip rather than emit
            # a null gene row - downstream we almost always want (disorder, gene)
            # pairs, and the full disorder list lives in product1.
            continue

        for assoc in assocs:
            gene = assoc.find("Gene")
            gene_symbol = _text(gene, "Symbol") if gene is not None else None
            gene_name = _text(gene, "Name") if gene is not None else None

            # Pivot gene external-references into a flat dict, one column per
            # well-known source.
            xref_flat: dict[str, str | None] = {c: None for c in GENE_XREF_SOURCES.values()}
            if gene is not None:
                for xref in gene.findall("ExternalReferenceList/ExternalReference"):
                    src = _text(xref, "Source")
                    if src in GENE_XREF_SOURCES:
                        xref_flat[GENE_XREF_SOURCES[src]] = _text(xref, "Reference")

            row = {
                "orpha_code": orpha_code,
                "disorder_name": disorder_name,
                "gene_symbol": gene_symbol,
                "gene_name": gene_name,
                "association_type": _text(assoc, "DisorderGeneAssociationType/Name"),
                "association_status": _text(assoc, "DisorderGeneAssociationStatus/Name"),
                "source_of_validation": _text(assoc, "SourceOfValidation"),
            }
            row.update(xref_flat)
            rows.append(row)

    df = pl.DataFrame(rows)

    # Cast numeric / categorical columns.
    df = df.with_columns(
        pl.col("orpha_code").cast(pl.Int64, strict=False),
        pl.col("association_type").cast(pl.Categorical),
        pl.col("association_status").cast(pl.Categorical),
    )

    return df


gene_df = parse_product6(gene_path)

n_disorders_g = gene_df.select(pl.col("orpha_code").n_unique()).item()
n_genes = gene_df.select(pl.col("gene_symbol").n_unique()).item()
print(f"Disorders with gene links : {n_disorders_g:,}")
print(f"Unique genes              : {n_genes:,}")
print(f"Total associations (rows) : {gene_df.shape[0]:,}")
print("\nSchema:")
for col, dtype in gene_df.schema.items():
    print(f"  {col:<22} {dtype}")
gene_df.head(5)

### 1.4 Save Parsed Tables to `data/`

We persist the parsed tables as **Parquet**. Parquet is a columnar, compressed binary format that is ~10× smaller than CSV for this data, preserves Polars dtypes (including `Categorical`), and is 20-50× faster to re-load than re-parsing the XML.

In [ ]:
XREF_PARQUET = DATA_DIR / "orphadata_disorder_xrefs.parquet"
GENE_PARQUET = DATA_DIR / "orphadata_disorder_genes.parquet"


def save_parquet(df: pl.DataFrame, path: Path) -> Path:
    """
    Write a DataFrame to Parquet and report file size.

    Parameters
    ----------
    df : pl.DataFrame
        Table to persist.
    path : Path
        Destination ``.parquet`` path.

    Returns
    -------
    Path
        The path that was written (for convenient chaining).
    """
    df.write_parquet(path)
    size_kb = path.stat().st_size / 1024
    print(f"Saved {df.shape[0]:>7,} rows -> {path}  ({size_kb:,.1f} KB)")
    return path


save_parquet(xref_df, XREF_PARQUET)
save_parquet(gene_df, GENE_PARQUET)

# Sanity check - re-read the Parquet files and confirm dtypes round-trip.
xref_check = pl.read_parquet(XREF_PARQUET)
gene_check = pl.read_parquet(GENE_PARQUET)
assert xref_check.shape == xref_df.shape, "disorder-xref round-trip shape mismatch"
assert gene_check.shape == gene_df.shape, "disorder-gene round-trip shape mismatch"
print("\nRound-trip check passed.")